# U05 應用開發（一）：sqlite3 深入・交易・Gradio 入門

**資料庫管理**・統計系三年級・10/08　<a href="https://colab.research.google.com/github/chang-ye-tu/db/blob/master/notebooks/unit05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

課程首頁：[github.com/chang-ye-tu/db](https://github.com/chang-ye-tu/db)・大綱：[syllabus.md](https://github.com/chang-ye-tu/db/blob/master/syllabus.md)・專題：[projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md)

今天把 SQL 接上 Python，做出**第一個會動的介面**——外加 ★專題說明會（兩支示範影片）

> **投影片式 notebook 使用法**：上課跟著往下走，程式格按 `Shift+Enter` 執行；左側「目錄」可跳節。回家可以重跑、改參數做實驗——**講義是可以跑的**。
>
> 開始前建議：檔案 → 在雲端硬碟中儲存副本，改動才會留下來。

## 0. 本單元地圖（135 分鐘）

| 節 | 分鐘 | 內容 |
|---|---|---|
| 第 1 節 | 50 | sqlite3 API 全套（Row／fetch 家族／executescript／date adapter）・**`?` 傳值的正常寫法**・交易（`with con:`／savepoint）・匯入效能・**萬列擬真資料合成七講究** |
| 第 2 節 | 50 | **Gradio**：Interface → Blocks・元件全覽・事件・`gr.update` 下拉刷新・`gr.State`・列選取編輯刪除・報表匯出・分頁・**第二個完整 app** |
| ★說明會 | 35 | `projects.md` 逐條・兩支示範影片・AI 協作工作流 |

**本單元是分水嶺**：前四個單元學「資料庫」，從今天起學「怎麼把資料庫**變成產品**」。

# 第 1 節：Python × SQLite 的正確接法

## 1.1 sqlite3 API 全套

```python
import sqlite3
con = sqlite3.connect("app.db")     # 連線（檔案不存在就建立）
cur = con.cursor()                  # 游標：執行與取結果的把手（con.execute 是捷徑，回傳 cursor）
cur.execute(sql, params)            # 執行一句（params 用 ? 把值交給 SQL，見 1.3）
cur.executemany(sql, seq)           # 批次（快非常多）
cur.fetchone() / fetchmany(k) / fetchall()   # 取一列 / k 列 / 全部
con.commit() / con.rollback()       # 確認 / 撤銷這批變更
con.close()
```

一個立刻讓生活變好的設定：**`row_factory`** 讓每列可以用**欄名**存取，不用記第幾欄。

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect("app.db")
con.execute("PRAGMA foreign_keys = ON")
con.row_factory = sqlite3.Row            # ← 關鍵設定：讓列變成「可用欄名存取」

con.executescript("""
DROP TABLE IF EXISTS member;
CREATE TABLE member(
  member_id INTEGER PRIMARY KEY,
  name  TEXT NOT NULL,
  dept  TEXT,
  balance INTEGER NOT NULL DEFAULT 0 CHECK (balance >= 0));
INSERT INTO member(name, dept, balance) VALUES
  ('林佳蓉','統計',80), ('陳威廷','統計',45), ('張雅筑','資訊',120), ('吳孟軒','數學',0);
""")
row = con.execute("SELECT * FROM member WHERE name = ?", ("林佳蓉",)).fetchone()
print("整列：", dict(row))
print("用欄名拿：", row["name"], "的餘額是", row["balance"])
print("不再需要 row[0]、row[3] 這種「猜第幾欄」的寫法")

In [ ]:
# fetch 家族與欄位資訊：一列、幾列、全部、逐列迭代——各有用途
cur = con.execute("SELECT member_id, name, balance FROM member ORDER BY member_id")
print("這次查詢的欄位：", [d[0] for d in cur.description])
print("fetchone()   →", dict(cur.fetchone()))            # 拿一列（游標往前走）
print("fetchmany(2) →", [dict(r) for r in cur.fetchmany(2)])   # 再拿兩列
print()
for r in con.execute("SELECT name FROM member"):          # 最 Python 的寫法：直接迭代
    print("逐列迭代：", r["name"])
print()
print("心法：介面要顯示表格 → pd.read_sql_query 一步到位；程式邏輯要判斷 → fetchone/迭代。")

In [ ]:
# 兩個常用的小情報：lastrowid（剛插入的自動編號）與 total_changes（這條連線改了幾列）
cur = con.execute("INSERT INTO member(name, dept) VALUES (?, ?)", ("賴品妍", "統計"))
print("剛插入那列的 member_id =", cur.lastrowid, "（開單→拿單號→寫明細，全靠它）")
n = con.execute("UPDATE member SET balance = balance + 10 WHERE dept = '統計'").rowcount
print("這句 UPDATE 改了", n, "列（rowcount）——「有沒有改到東西」是業務判斷的重要訊號")
print("連線累計異動：", con.total_changes, "列")
con.commit()

In [ ]:
# executescript：一次跑一整段 SQL 腳本（建表、種子資料的好朋友）
con.executescript("""
DROP TABLE IF EXISTS announcement;
CREATE TABLE announcement(
  id   INTEGER PRIMARY KEY,
  body TEXT NOT NULL,
  at   TEXT NOT NULL DEFAULT (datetime('now','localtime')));
INSERT INTO announcement(body) VALUES ('上課地點改電腦教室 2'), ('專題說明會在今天第三節');
""")
print(con.execute("SELECT COUNT(*) FROM announcement").fetchone()[0], "則公告")
print("兩個注意：① executescript 執行前會把「目前未 commit 的交易」先 commit——所以它只放 setup；")
print("          ② 腳本裡不能用 ?（它是純 SQL 批次）。日常操作仍然 execute ＋ 佔位符。")

## 1.2 date adapter／converter：讓日期「進出自動翻譯」

日期在 SQLite 裡存 ISO 字串（U02 的慣例），撈回 Python 是 `str`。想讓它**存進去自動轉字串、撈回來自動變 `date` 物件**？註冊一對翻譯員即可。（Python 3.12 起內建的預設翻譯員已標記棄用——**自己註冊**是正規做法，AI 還常給你舊寫法，看到就改。）

In [ ]:
import datetime

sqlite3.register_adapter(datetime.date, lambda d: d.isoformat())            # date → 存進去的樣子
sqlite3.register_converter("DATE", lambda b: datetime.date.fromisoformat(b.decode()))  # 撈出來 → date

dcon = sqlite3.connect("dates.db", detect_types=sqlite3.PARSE_DECLTYPES)    # 認得宣告型別才會翻譯
dcon.execute("DROP TABLE IF EXISTS loan_demo")
dcon.execute("CREATE TABLE loan_demo(book TEXT, due DATE)")                 # 欄位宣告 DATE
dcon.execute("INSERT INTO loan_demo VALUES (?, ?)", ("統計學習導論", datetime.date(2026, 10, 22)))
dcon.commit()

book, due = dcon.execute("SELECT book, due FROM loan_demo").fetchone()
print(f"{book} 到期日 {due}（型別：{type(due).__name__}）")
print("拿回來就是 date 物件，直接算：還有", (due - datetime.date(2026, 10, 8)).days, "天")
dcon.close()
# 不想用翻譯員也行：存 ISO 字串、要算時 fromisoformat()——兩派都對，專題選一派用到底

## 1.3 把「值」交給 SQL 的正常寫法：`?` 佔位符

SQL 字串裡凡是「執行時才知道的值」，一律寫 `?`，值另外用 tuple 傳：

```python
con.execute("SELECT * FROM member WHERE name = ?", (name,))
```

為什麼不自己用 f-string 把值拼進字串？三個很實際的理由：

1. **引號會絆倒你**：值裡有 `'`（書名、外文人名、任何自由輸入）字串就拼壞了（下一格現場示範）；
2. **型別要自己伺候**：日期要不要引號？`None` 怎麼寫成 `NULL`？`?` 讓 sqlite3 替你把每種值擺對；
3. **executemany 就是它**：等下合成萬列資料，非它不可。

口訣：**SQL 管句型，`?` 管值**。

In [ ]:
# 引號絆倒示範：值裡帶一個 '，手拼字串當場跌倒
title_str = "Fisher's Exact Test 傳奇"        # 合法的書名，中間有個 '
con.execute("DROP TABLE IF EXISTS book_demo")
con.execute("CREATE TABLE book_demo(title TEXT NOT NULL)")

try:
    con.execute(f"INSERT INTO book_demo VALUES ('{title_str}')")   # 手拼：引號在 Fisher 後面就斷句了
except sqlite3.OperationalError as e:
    print("手拼字串跌倒 →", e)

con.execute("INSERT INTO book_demo VALUES (?)", (title_str,))      # ? 版：毫髮無傷
con.commit()
print("? 版寫入成功 →", con.execute("SELECT title FROM book_demo").fetchone()[0])
print("日期、None、含逗號的地址⋯⋯同理。值一律走 ?，從此不用想引號的事。")

In [ ]:
# 佔位符第二型：具名（:名字 ＋ dict）——欄位一多，可讀性差很多
con.execute("DROP TABLE IF EXISTS order_demo")
con.execute("CREATE TABLE order_demo(who TEXT, item TEXT, qty INTEGER, note TEXT)")

form = {"who": "林佳蓉", "item": "珍珠奶茶", "qty": 2, "note": None}      # 想像這包是 UI 表單收進來的
con.execute("INSERT INTO order_demo VALUES (:who, :item, :qty, :note)", form)
con.commit()
print(dict(con.execute("SELECT * FROM order_demo").fetchone()))
print("→ None 自動變 NULL；之後 Gradio 表單收進來就是一包欄位值，dict 直接對走。")

In [ ]:
# IN 清單的長度執行時才知道 → 產生「剛好數量」的 ?，值照樣全走佔位符
ids = [1, 3]
ph = ",".join("?" * len(ids))                        # "?,?"
sql = f"SELECT member_id, name FROM member WHERE member_id IN ({ph})"
print("組出來的 SQL：", sql)
print([tuple(r) for r in con.execute(sql, ids)])

In [ ]:
# ? 只能代「值」——欄名／表名是句型的一部分，要動態就用「白名單」挑好再組
def list_members_sorted(sort_col):
    if sort_col not in {"name", "balance", "member_id"}:   # 只能從白名單挑
        sort_col = "member_id"
    return pd.read_sql_query(
        f"SELECT member_id, name, balance FROM member ORDER BY {sort_col} DESC", con)

print(list_members_sorted("balance").to_string(index=False))
print()
print(list_members_sorted("亂七八糟 DROP TABLE").to_string(index=False))   # 不在白名單 → 落回預設欄
print("→ 使用者能影響的永遠只有「白名單裡挑哪個」；句型自始至終是你寫死的。")

## 1.4 交易：把「必須全對或全不動」的操作綁在一起

經典例子：轉帳——扣款和入帳必須**同生共死**。中間出錯而只扣沒入，錢就蒸發了。

In [ ]:
# 轉帳：兩步驟必須原子完成
def transfer(src, dst, amt):
    try:
        with con:                                    # ← with 區塊：正常結束自動 commit，出例外自動 rollback
            con.execute("UPDATE member SET balance = balance - ? WHERE name = ?", (amt, src))
            con.execute("UPDATE member SET balance = balance + ? WHERE name = ?", (amt, dst))
        return "✅ 轉帳成功"
    except sqlite3.IntegrityError as e:
        return f"❌ 交易被撤銷（餘額不足，balance>=0 擋下）→ 資料回到轉帳前，沒有半途而廢：{e}"

def balances():
    return {r["name"]: r["balance"] for r in con.execute("SELECT name, balance FROM member")}

print("轉帳前：", balances())
print(transfer("林佳蓉", "張雅筑", 30));  print("之後：", balances())
print(transfer("張雅筑", "林佳蓉", 999));  print("之後：", balances(), "← 失敗那筆完全沒動到餘額（原子性）")

In [ ]:
# 對照組：沒有交易的世界——兩步之間程式炸掉，錢蒸發在半路
def transfer_risky(src, dst, amt, boom=False):
    con.execute("UPDATE member SET balance = balance - ? WHERE name = ?", (amt, src))
    if boom:
        raise RuntimeError("程式在兩步之間掛了（想像：斷線、bug、Colab runtime 被回收）")
    con.execute("UPDATE member SET balance = balance + ? WHERE name = ?", (amt, dst))
    con.commit()

print("事前：", balances())
try:
    transfer_risky("林佳蓉", "張雅筑", 10, boom=True)
except RuntimeError as e:
    print("💥", e)
print("事後：", balances(), "← 10 元卡在半路（扣了沒加）！")
con.rollback()
print("rollback 撿回：", balances())
print("→ 這次僥倖是因為還沒 commit。with con: 的價值：把兩步綁成一步，炸掉自動回滾，不用人肉善後。")

### `with con:` 到底做了什麼？

- 進入區塊 → 開始交易；區塊正常結束 → `commit()`；區塊內拋例外 → `rollback()` 後把例外往外丟。
- 這是專題「必須原子完成的業務操作」（共同要求第 4 條）的標準寫法：扣名額、借出、成交、扣庫存⋯⋯全部包在 `with con:` 裡。
- ⚠️ 陷阱：`with con:` 管的是**交易**，不是連線；它**不會** close 連線。別跟 `with open()` 搞混。

（並行競態——兩個人同時搶最後一個名額——是 U06 的主題，那裡教 `BEGIN IMMEDIATE` 與條件式 UPDATE。）

### savepoint：交易裡的「存檔點」

整批匯入 3 包資料，第 2 包壞掉——全部回滾太浪費、硬塞又留髒資料？
**savepoint ＝ 巢狀的小交易**：壞的那段回到存檔點退掉，好的段落照常保留。

In [ ]:
# savepoint 實戰：三包資料、一包有毒——壞包整包退、好包全保留
batches = [[("A1", 100), ("A2", 200)],
           [("B1", 300), ("B2", -999)],        # ← 有毒：金額違反 CHECK
           [("C1", 500)]]
con.execute("DROP TABLE IF EXISTS bulk_import")
con.execute("CREATE TABLE bulk_import(code TEXT PRIMARY KEY, amt INTEGER CHECK (amt >= 0))")

with con:                                            # 外層大交易
    for i, batch in enumerate(batches, 1):
        con.execute(f"SAVEPOINT sp{i}")              # 立存檔點（名稱是識別字，不是值）
        try:
            con.executemany("INSERT INTO bulk_import VALUES (?,?)", batch)
            con.execute(f"RELEASE sp{i}")
            print(f"第 {i} 包：✅ 寫入 {len(batch)} 筆")
        except sqlite3.IntegrityError as e:
            con.execute(f"ROLLBACK TO sp{i}")        # 只退這一包
            con.execute(f"RELEASE sp{i}")
            print(f"第 {i} 包：❌ 整包退回（{e}）")

print("最後留下：", con.execute("SELECT COUNT(*) FROM bulk_import").fetchone()[0],
      "筆（期望 3：好包保留、壞包乾淨退掉）")

### 隨堂練習 A（5 分鐘）：寫一個有交易保護的 `charge()`

規格：`charge(name, amt)` 扣款——① 用 `with con:` 保護；② 餘額不足時回傳友善訊息（不能讓例外炸出去）；
③ 查無此人也要有訊息；④ 配 **2 個 assert**（一個成功、一個「應該失敗」）。寫在下一格。

<details><summary>參考解</summary>

```python
def charge(name, amt):
    try:
        with con:
            n = con.execute("UPDATE member SET balance = balance - ? WHERE name = ?",
                            (amt, name)).rowcount
            if n == 0:
                raise ValueError("查無此人")           # 在 with 裡 raise → 自動回滾
        return f"✅ 已扣 {amt} 元"
    except sqlite3.IntegrityError:
        return "❌ 餘額不足，這筆取消"
    except ValueError as e:
        return f"⚠️ {e}"

assert charge("林佳蓉", 10).startswith("✅")
assert charge("林佳蓉", 99999).startswith("❌")      # 應該失敗的測試
assert charge("路人甲", 10).startswith("⚠️")
```
這就是專題共同要求第 8 條「≥8 個 assert、含應該失敗」的長相——每寫一個函數就順手配測試。
</details>

In [ ]:
# 練習 A 工作區
def charge(name, amt):
    ...   # TODO





## 1.5 匯入效能：executemany 與交易的威力

In [ ]:
import time
con.execute("DROP TABLE IF EXISTS bulk"); con.execute("CREATE TABLE bulk(x INTEGER)")
rows = [(i,) for i in range(50_000)]

t = time.time()                                          # ❌ 每筆一個交易（每次都 fsync 落地）
for r in rows[:5000]:
    con.execute("INSERT INTO bulk VALUES (?)", r); con.commit()
t_slow = time.time() - t

con.execute("DELETE FROM bulk")
t = time.time()                                          # ✅ 一個交易 + executemany
with con:
    con.executemany("INSERT INTO bulk VALUES (?)", rows)
t_fast = time.time() - t

print(f"逐筆 commit 5,000 列：{t_slow:.3f}s")
print(f"executemany 50,000 列（10 倍量）：{t_fast:.3f}s")
print(f"→ 換算單列，executemany 快約 {(t_slow/5000)/(t_fast/50000):.0f} 倍。合成萬列資料時這是天與地的差別。")

In [ ]:
# DataFrame 進資料庫的捷徑：to_sql（底層也是批次插入）——合成資料收工就用它
df_bulk = pd.DataFrame({"x": range(50_000)})
t = time.time()
df_bulk.to_sql("bulk_pd", con, if_exists="replace", index=False)
print(f"pandas to_sql 50,000 列：{time.time()-t:.3f}s")
print("→ numpy/pandas 造好資料 → to_sql 一行入庫；之後查詢照常用 SQL。")

In [ ]:
# 批次匯入的好搭檔：UPSERT（U02 教過）× executemany——重複名單「刷新」而不是報錯
con.execute("DROP TABLE IF EXISTS roster")
con.execute("CREATE TABLE roster(sid TEXT PRIMARY KEY, name TEXT, credits INTEGER)")
week1 = [("S001", "林佳蓉", 3), ("S002", "陳威廷", 3)]
week2 = [("S002", "陳威廷", 6), ("S003", "張雅筑", 3)]    # S002 重複出現（學分要更新）
for wk in (week1, week2):
    with con:
        con.executemany("""INSERT INTO roster VALUES (?,?,?)
                            ON CONFLICT(sid) DO UPDATE SET credits = excluded.credits""", wk)
print(pd.read_sql_query("SELECT * FROM roster ORDER BY sid", con).to_string(index=False))
print("→ 每週重匯名單不炸 PK、舊資料自動刷新——匯入類功能（名單、庫存盤點）的標準組合。")

## 1.6 合成擬真資料：你的專題沒有真使用者

專題要 ≥ 10,000 列，且分佈要「像真的」。**七個統計系該有的講究**——前三個是基本盤：

In [ ]:
import numpy as np
rng = np.random.default_rng(20261008)         # 固定 seed → 人人可重現（專題要求）

N = 12_000
# 講究 1) 長尾熱門度：少數商品/活動佔大多數流量（Zipf）——別用均勻分佈，真實世界不長那樣
pid = rng.zipf(1.4, N * 3)
pid = pid[pid <= 200][:N]                       # 砍掉尾巴太大的編號，保留 N 筆
# 講究 2) 時間有節律：週末下單多（給一年 365 天各自的權重，週五六日加碼）
week_w = np.tile([1, 1, 1, 1, 1.4, 1.8, 1.6], 53)[:365]   # 週一..週日 的相對量，鋪滿 365 天
week_w = week_w / week_w.sum()                  # 正規化成機率
day    = rng.choice(365, N, p=week_w)           # 依權重抽「第幾天」
odate  = (np.datetime64("2026-01-01") + day.astype("timedelta64[D]")).astype(str)
# 講究 3) 金額右偏：lognormal（不是常態！價格、金額幾乎都右偏，少數大單拉高平均）
amount = np.round(rng.lognormal(4.5, 0.6, N)).astype(int) + 10

demo = pd.DataFrame({"pid": pid, "odate": odate, "amount": amount})
print(demo.describe(include="all").loc[["count","mean","min","max"]].round(1).to_string())
print("\n熱門度前 5 名商品佔比：",
      round(demo.pid.value_counts(normalize=True).head(5).sum()*100, 1), "% ← 長尾效應")
print("金額中位數 vs 平均：", int(demo.amount.median()), "vs", int(demo.amount.mean()),
      "→ 右偏（平均被大單拉高）")

In [ ]:
# 講究 4) 欄位之間要有合理的「相關」——年齡影響客單價（條件分佈：統計系的母語）
age = rng.integers(18, 66, N)
cond_mean = 250 + 6 * (age - 18)                            # 年齡越大、客單越高的線性趨勢
amount2 = np.round(rng.lognormal(np.log(cond_mean), 0.35)).astype(int)   # 繞著條件平均的右偏雜訊
corr_demo = pd.DataFrame({"age": age, "amount": amount2})
print(corr_demo.groupby(pd.cut(corr_demo.age, [17, 30, 45, 65]), observed=True)
      .amount.agg(["count", "mean"]).round(0).to_string())
print("\n→ 有相關，報表「年齡層 × 消費」才有故事可講；全獨立亂數的資料，圖表平到說不出話。")

In [ ]:
#@title 🈶 圖表中文字型（Colab 需安裝一次；本機有 Noto 就直接生效）
import os, glob, subprocess, matplotlib
from matplotlib import font_manager

cjk = sorted(glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"))
if not cjk:                                    # Colab 第一次：裝字型（約 10 秒）
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-cjk"], capture_output=True)
    cjk = sorted(glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"))
if cjk:
    font_manager.fontManager.addfont(cjk[0])
    matplotlib.rcParams["font.family"] = font_manager.FontProperties(fname=cjk[0]).get_name()
    print("中文字型就緒 ✅：", os.path.basename(cjk[0]))
else:
    print("⚠️ 找不到 CJK 字型——圖表中文會變 □（不影響其他功能）")
matplotlib.rcParams["axes.unicode_minus"] = False

In [ ]:
# 講究 5) 一天之內也有節律：手搖店的訂單擠在午、晚餐尖峰（小時權重抽時間戳）
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

hour_w = np.array([0,0,0,0,0,0, 0,1,2,3,5,9, 14,10,6,5,6,10, 13,8,4,2,1,0], dtype=float)
hour   = rng.choice(24, N, p=hour_w / hour_w.sum())
ts_list = [f"2026-10-{d:02d} {h:02d}:{m:02d}"
           for d, h, m in zip(rng.integers(1, 32, N), hour, rng.integers(0, 60, N))]
print("長相：", ts_list[:3])

fig, ax = plt.subplots(figsize=(8, 2.4))
pd.Series(hour).value_counts().sort_index().plot.bar(ax=ax, width=0.85)
ax.set_xlabel("時段"); ax.set_ylabel("訂單數"); ax.set_title("各時段訂單量（合成資料）")   # 中文標籤：上一格字型已就緒
plt.tight_layout(); plt.show()
# 你的題目的節律是什麼？健身房＝下班尖峰、圖書館＝考前爆量、民宿＝週末與連假——寫進生成假設

In [ ]:
# 講究 6) 假得像真的：姓名與地址產生器（同 seed 同名單；重名是特色，真實世界也重名）
SURNAMES = list("陳林黃張李王吳劉蔡楊許鄭謝郭洪曾邱廖賴周")
GIVEN = ["佳蓉","威廷","雅筑","承翰","思穎","冠宇","孟軒","子涵","明修","芷瑄",
         "宇翔","欣妤","偉倫","品妍","柏勳","韻如","家豪","美慧","國彬","語彤"]
ROADS = ["文華路","河南路","逢甲路","福星路","西屯路","青海路","漢口路","台灣大道"]

def fake_people(n, r):
    return pd.DataFrame({
        "name": [r.choice(SURNAMES) + r.choice(GIVEN) for _ in range(n)],
        "addr": [f"台中市西屯區{r.choice(ROADS)}{r.integers(1, 500)}號" for _ in range(n)]})

print(fake_people(5, np.random.default_rng(20261008)).to_string(index=False))
print(f"\n姓 {len(SURNAMES)} × 名 {len(GIVEN)} = {len(SURNAMES)*len(GIVEN)} 種組合，夠專題用；"
      "電話用 f'09{r.integers(10**8):08d}'。")

In [ ]:
# 講究 7) 故意弄髒一點：真實資料有缺漏——注入少量 NULL 與怪值，你的報表與清理才有戲
dirty = corr_demo.copy()
null_idx  = rng.choice(N, int(N * 0.03), replace=False)      # 3% 年齡缺失
weird_idx = rng.choice(N, 5, replace=False)                  # 5 筆離譜金額（輸入錯誤）
dirty.loc[null_idx, "age"] = np.nan
dirty.loc[weird_idx, "amount"] = dirty.loc[weird_idx, "amount"] * 100
print(f"缺失年齡：{dirty.age.isna().sum()} 筆（{dirty.age.isna().mean()*100:.1f}%）")
print(f"金額前 3 大：{sorted(dirty.amount, reverse=True)[:3]} ← 離群值，報表該怎麼對待它們？")
print("→ 完美無缺的資料反而假。注入了什麼髒東西、報表怎麼處理（排除？標註？），都寫進生成假設。")
print("  （比例拿捏：3–5% 缺失就夠說故事，別把資料弄爛到自己分析不動。）")

In [ ]:
# 把合成包成「吃 seed 的函數」——同題同學一人一個 seed，資料自然不同（專題規定）
def synth_orders(n, seed):
    r = np.random.default_rng(seed)
    amt = np.round(r.lognormal(4.3, 0.5, n)).astype(int) + 10
    pid = r.zipf(1.5, n * 3); pid = pid[pid <= 50][:n]
    return pd.DataFrame({"pid": pid, "amount": amt})

for seed in (101, 202, 303):
    d = synth_orders(1000, seed)
    print(f"seed={seed}：平均客單 {d.amount.mean():6.1f}、最熱商品被點 {d.pid.value_counts().iloc[0]} 次")
print("→ 同一支程式、不同 seed → 三份不同但同分佈的資料。seed 與生成假設都寫進你的 notebook。")

> **📌 合成資料的心法**：你在替一個「想像中的系統」造它的歷史。想清楚——你的使用者是誰？
> 行為有什麼規律（尖峰時段、熱門品項、季節性、少數重度使用者、欄位間的相關）？把這些寫進分佈，
> 你的報表才有故事可講。**這一步本身就是統計建模**，報告裡要說明你的生成假設。

### 隨堂練習 B（4 分鐘）：驗收自己的合成資料

拿剛剛的 `demo` DataFrame，用兩張圖自我驗收：① `amount` 的直方圖（右偏了嗎？）
② 每週各天的訂單數長條圖（週末有比較高嗎？提示：`pd.to_datetime(demo.odate).dt.dayofweek`）。

<details><summary>參考解</summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(demo.amount, bins=60)
axes[0].set_title("金額分佈（右偏）")
dow = pd.to_datetime(demo.odate).dt.dayofweek.value_counts().sort_index()
axes[1].bar(dow.index, dow.values)
axes[1].set_title("週一(0)～週日(6) 訂單數")
plt.tight_layout(); plt.show()
```
生成假設 → 畫圖驗證 → 不像就調參數。這個迴圈就是「合成資料的品管」，報告放這兩張圖很加分。
</details>

In [ ]:
# 練習 B 工作區
# TODO




# 第 2 節：Gradio——兩行程式長出一個網頁介面

## 2.1 為什麼是 Gradio？

| | 為什麼適合本課 |
|---|---|
| 零前端知識 | 不用學 HTML/CSS/JS，純 Python |
| Colab 原生 | `launch()` 直接內嵌在 notebook，教室零安裝 |
| AI 友善 | 語法簡單規律，AI 生成品質高 |
| 能分享 | `share=True` 給你一個 72 小時公開網址，報告時同學掃 QR 就能玩 |

安裝（每個新 runtime 一次）：`!pip install -q gradio`，約 30 秒。

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("gradio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"])
import gradio as gr
print("gradio", gr.__version__, "就緒")

## 2.2 最小範例：Interface（一個函數 = 一個介面）

`gr.Interface` 的三要素：`fn`（處理函數）、`inputs`（輸入元件）、`outputs`（輸出元件）。
使用者填輸入 → Gradio 呼叫你的 `fn` → 顯示回傳值。就這樣。

In [ ]:
# 最小範例：BMI 計算機（先感受「函數→介面」的對應）
def bmi(height_cm, weight_kg):
    h = height_cm / 100
    b = weight_kg / (h * h)
    label = "過輕" if b < 18.5 else "正常" if b < 24 else "過重" if b < 27 else "肥胖"
    return f"BMI = {b:.1f}（{label}）"

demo_bmi = gr.Interface(
    fn=bmi,
    inputs=[gr.Number(label="身高 (cm)", value=170), gr.Number(label="體重 (kg)", value=65)],
    outputs=gr.Textbox(label="結果"),
    examples=[[170, 65], [158, 45]],          # 範例列：使用者一鍵帶入（demo 時超好用）
    title="BMI 計算機", flagging_mode="never")
print("介面建好；下一格 launch")

In [ ]:
# [SKIP-TEST] 在 Colab 執行會內嵌出現可操作的介面
demo_bmi.launch(height=420)

### 隨堂練習 C（5 分鐘）：幫 BMI 加一個輸入

把上面的 BMI 介面加一個「性別」`gr.Radio(["男","女"])` 輸入，結果字串裡帶上它（例：`女生 BMI = 20.3（正常）`）。
改兩個地方：函數簽名、inputs 清單。改完 launch 玩玩看。

<details><summary>參考解</summary>

```python
def bmi2(gender, height_cm, weight_kg):
    h = height_cm / 100
    b = weight_kg / (h * h)
    label = "過輕" if b < 18.5 else "正常" if b < 24 else "過重" if b < 27 else "肥胖"
    return f"{gender}生 BMI = {b:.1f}（{label}）"

gr.Interface(fn=bmi2,
             inputs=[gr.Radio(["男", "女"], value="女", label="性別"),
                     gr.Number(label="身高 (cm)", value=170), gr.Number(label="體重 (kg)", value=65)],
             outputs=gr.Textbox(label="結果"), flagging_mode="never").launch(height=420)
```
規律看出來了嗎？**inputs 的順序 ＝ 函數參數的順序**——Gradio 的一切都是這條規律。
</details>

In [ ]:
# 練習 C 工作區
# TODO




## 2.3 元件全覽：每個元件對應一種「資料庫欄位」

| 元件 | 收什麼 | 資料庫對應 |
|---|---|---|
| `gr.Textbox` | 自由文字 | TEXT |
| `gr.Number` | 數字 | INTEGER／REAL |
| `gr.Slider(a, b, step)` | 有範圍的數值 | 帶 CHECK 範圍的數值欄 |
| `gr.Dropdown(choices)` | 從清單選一個 | **FK 欄位**／`CHECK (IN ...)` 欄位 |
| `gr.Radio(choices)` | 少量互斥選項 | 狀態、類別欄 |
| `gr.Checkbox` | 勾／不勾 | 布林 0/1 |
| `gr.Dataframe` | 顯示表格 | 查詢結果（`pd.read_sql_query`） |
| `gr.Plot` | 顯示圖 | matplotlib figure |
| `gr.State` | **不顯示**的暫存值 | 還沒進資料庫的東西（購物籃） |

**黃金心法：`Dropdown` 的 `choices` 從資料庫撈**——這就是 FK 欄位的標準介面（等下 2.6 示範「新資料進來，選單自動長」）。

In [ ]:
# 元件圖鑑：一次全擺出來認臉（下一格 launch 看全家福）
with gr.Blocks() as widget_zoo:
    gr.Markdown("### 元件圖鑑——每個都對應一種資料庫欄位")
    with gr.Row():
        gr.Textbox(label="Textbox → TEXT")
        gr.Number(label="Number → INTEGER/REAL", value=0)
    with gr.Row():
        gr.Slider(1, 20, value=1, step=1, label="Slider → 帶範圍的數值（CHECK 1–20）")
        gr.Dropdown(["統計", "資訊", "數學", "企管"], label="Dropdown → FK／CHECK IN")
    with gr.Row():
        gr.Radio(["正常", "半糖", "微糖", "無糖"], value="正常", label="Radio → 狀態/類別欄")
        gr.Checkbox(label="Checkbox → 布林 0/1（已繳費？）")
    gr.Dataframe(label="Dataframe → 查詢結果", value=pd.DataFrame({"示範": ["查詢結果顯示在這"]}))
print("元件展示建好")

In [ ]:
# [SKIP-TEST]
widget_zoo.launch(height=620)

## 2.4 接上資料庫：查詢介面（回傳 DataFrame → `gr.Dataframe`）

真正的應用是「介面 ↔ 資料庫」。兩條紀律先立好：

1. **連線加 `check_same_thread=False`**：Gradio 會在別的執行緒呼叫你的函數，sqlite3 預設不准跨執行緒用同一條連線——教學應用開這個參數即可（全 app 就用這一條連線，正式服務才需要更講究的連線管理）。
2. **UI 薄、邏輯厚**：處理函數只做「收參數 → 查/改資料庫 → 回傳」，**可以脫離 UI 單獨測試**（這也是專題測試的對象）。

In [ ]:
# 為 UI 準備連線
gcon = sqlite3.connect("app.db", check_same_thread=False)
gcon.row_factory = sqlite3.Row
gcon.execute("PRAGMA foreign_keys = ON")

# 後端邏輯：純函數，先單獨測試（這就是「UI 薄、邏輯厚」）
def search_members(keyword, min_balance):
    return pd.read_sql_query(
        """SELECT member_id AS 編號, name AS 姓名, dept AS 系所, balance AS 餘額
           FROM member
           WHERE name LIKE ? AND balance >= ?
           ORDER BY balance DESC""",
        gcon, params=(f"%{keyword}%", min_balance))

print("後端函數單獨測試（不經 UI）：")
print(search_members("", 0).to_string(index=False))
assert len(search_members("林", 0)) == 1        # 可以寫 assert → 這就是專題要求的測試
print("\n✅ 後端測試通過，才接 UI")

In [ ]:
# 前端：把函數包成介面
demo_query = gr.Interface(
    fn=search_members,
    inputs=[gr.Textbox(label="姓名關鍵字（空白=全部）"),
            gr.Slider(0, 100, value=0, step=10, label="最低餘額")],
    outputs=gr.Dataframe(label="查詢結果"),
    title="會員查詢", flagging_mode="never")
print("查詢介面建好；下一格 launch")

In [ ]:
# [SKIP-TEST]
demo_query.launch(height=460)

## 2.5 Blocks：自由排版 ＋ 多元件互動（專題主力）

`Interface` 適合「一函數一介面」；真正的應用要用 **`gr.Blocks`**——自由擺放、多顆按鈕、分頁、事件綁定。

心智模型：
```python
with gr.Blocks() as demo:
    widget = gr.Textbox(...)        # 宣告元件
    btn    = gr.Button("送出")
    out    = gr.Dataframe()
    btn.click(fn=handler, inputs=[widget], outputs=[out])   # 綁定：按下→呼叫 fn→更新輸出
```
`inputs` 的目前值餵給 `fn`，`fn` 的回傳依序更新 `outputs`——**幾個輸出就回傳幾個值**。

In [ ]:
# 事件三兄弟：click（按鈕）/ change（值一變就觸發）/ submit（在輸入框按 Enter）
with gr.Blocks() as event_demo:
    t = gr.Textbox(label="打字看看（change 每次變動都觸發；Enter＝submit）")
    b = gr.Button("或按我（click）")
    o = gr.Textbox(label="回音", interactive=False)

    def echo(s):
        return f"你說：{s}"

    t.change(echo, t, o)      # 即時搜尋用它
    t.submit(echo, t, o)      # 「打完按 Enter」用它
    b.click(echo, t, o)       # 明確動作（新增、刪除）用它
print("同一個後端函數可以掛在多種事件上——事件選擇＝操作手感的設計")

In [ ]:
# Blocks 實戰：一個「新增會員 + 即時列表」的迷你後台（含表單驗證與錯誤回饋）
def add_member(name, dept, balance):
    name = (name or "").strip()
    if not name:
        return "⚠️ 姓名必填", list_members()               # 驗證：友善錯誤，不讓例外炸到使用者臉上
    try:
        with gcon:
            gcon.execute("INSERT INTO member(name, dept, balance) VALUES (?,?,?)",
                         (name, dept or None, int(balance)))
        return f"✅ 已新增「{name}」", list_members()
    except sqlite3.IntegrityError as e:
        return f"❌ 新增失敗：{e}", list_members()

def list_members():
    return pd.read_sql_query("SELECT member_id AS 編號, name AS 姓名, dept AS 系所, balance AS 餘額 "
                             "FROM member ORDER BY member_id", gcon)

with gr.Blocks(title="會員後台") as admin_ui:
    gr.Markdown("## 會員管理後台")
    with gr.Row():
        with gr.Column(scale=1):
            in_name = gr.Textbox(label="姓名（必填）")
            in_dept = gr.Dropdown(["統計","資訊","數學","企管"], label="系所")
            in_bal  = gr.Number(label="初始餘額", value=0)
            btn     = gr.Button("新增", variant="primary")
            msg     = gr.Textbox(label="訊息", interactive=False)
        with gr.Column(scale=2):
            table_out = gr.Dataframe(value=list_members(), label="目前會員")
    btn.click(add_member, inputs=[in_name, in_dept, in_bal], outputs=[msg, table_out])

# 後端邏輯照樣先測（不經 UI）
n0 = len(list_members()); add_member("測試員", "統計", 5)
assert len(list_members()) == n0 + 1
print(f"✅ 新增邏輯測試通過（{n0} → {len(list_members())} 列）；下一格 launch 看介面")

In [ ]:
# [SKIP-TEST] share=True 會多給一個公開網址（報告時同學可用手機操作）
admin_ui.launch(height=520)   # 正式報告可用 admin_ui.launch(share=True)

### 隨堂練習 D（4 分鐘）：查詢介面加一個「系所」過濾

把 2.4 的 `search_members` 加第三個參數 `dept`（`gr.Dropdown(["全部","統計","資訊","數學","企管"])`）：
選「全部」不過濾，否則 `AND dept = ?`。想一下：SQL 要怎麼寫才能「一句話兩用」？

<details><summary>參考解（兩個常見寫法）</summary>

```python
def search_members2(keyword, min_balance, dept):
    sql = """SELECT member_id, name, dept, balance FROM member
             WHERE name LIKE ? AND balance >= ?
               AND (? = '全部' OR dept = ?)          -- 一句兩用：參數自己當開關
             ORDER BY balance DESC"""
    return pd.read_sql_query(sql, gcon, params=(f"%{keyword}%", min_balance, dept, dept))
```
或者在 Python 端組（條件多時更清楚）：`if dept != "全部": sql += " AND dept = ?"; params.append(dept)`——
句型仍是寫死的分支，值仍走 `?`。表單一多，這兩招是所有「進階搜尋」頁的骨架。
</details>

In [ ]:
# 練習 D 工作區
# TODO




## 2.6 三個進階招式：`gr.update`・`gr.State`・列選取

這三招補齊 CRUD 介面的最後一哩：**下拉選單跟著資料長**（U 的前提是「選得到」）、**還沒進庫的暫存**（購物籃）、**點表格選列**（U 與 D 的入口）。

In [ ]:
# 招式一 gr.update：新增會員後，Dropdown 的選項立刻長出來——FK 下拉的招牌 pattern
def member_dd():
    names = [r["name"] for r in gcon.execute("SELECT name FROM member ORDER BY member_id")]
    return gr.update(choices=names, value=None)      # 回傳「更新元件屬性」的指令

def add_and_refresh(name):
    msg, tbl = add_member(name, None, 0)
    return msg, tbl, member_dd()                      # 第三個輸出：更新下拉選單

with gr.Blocks() as refresh_demo:
    with gr.Row():
        new_name = gr.Textbox(label="新會員姓名")
        dd = gr.Dropdown(choices=[], label="選一位會員（新增後自動出現）")
    msg = gr.Textbox(label="訊息", interactive=False)
    tbl = gr.Dataframe(value=list_members())
    gr.Button("新增", variant="primary").click(add_and_refresh, [new_name], [msg, tbl, dd])
    refresh_demo.load(member_dd, outputs=dd)          # 頁面載入時先撈一次資料庫

print("✅ pattern：任何會改資料的事件，把「重撈清單」也放進 outputs——畫面永遠跟資料庫同步")

In [ ]:
# 招式二 gr.State：購物籃——「還沒送出的東西」不進資料庫，先放 State
MENU = {"珍珠奶茶": 65, "烏龍綠": 35, "紅茶拿鐵": 55}

def add_item(drink, basket):
    basket = basket + [(drink, MENU[drink])]          # State 就是普通參數：傳進來、改一改、傳回去
    df = pd.DataFrame(basket, columns=["品項", "價格"])
    return basket, df, f"小計 {df["價格"].sum()} 元"

def clear_basket():
    return [], pd.DataFrame(columns=["品項", "價格"]), "小計 0 元"

with gr.Blocks() as order_ui:
    basket = gr.State([])                              # 每個使用者自己的暫存（不顯示、不進庫）
    dd_drink = gr.Dropdown(list(MENU), value="珍珠奶茶", label="飲料")
    tbl = gr.Dataframe(label="購物籃")
    subtotal = gr.Textbox(interactive=False, label="金額")
    with gr.Row():
        gr.Button("加入").click(add_item, [dd_drink, basket], [basket, tbl, subtotal])
        gr.Button("清空").click(clear_basket, None, [basket, tbl, subtotal])

b, df, s = add_item("烏龍綠", []); b, df, s = add_item("珍珠奶茶", b)     # 後端照樣先測
assert s == "小計 100 元" and len(b) == 2
print("✅ State 邏輯測試通過——按「送出訂單」時才把整籃用一個交易寫進資料庫（點餐類系統的核心流程）")

### 隨堂練習 E（5 分鐘）：幫點餐籃加「送出訂單」

規格：一顆「送出訂單」按鈕——把 `gr.State` 裡的整籃，用**一個交易**寫進兩張表
（`orders(order_id, at)`＋`order_items(order_id, item, price)`），成功後清空籃子並顯示訂單編號。

<details><summary>參考解（骨架）</summary>

```python
ocon = sqlite3.connect("orders.db", check_same_thread=False)
ocon.executescript("""
CREATE TABLE IF NOT EXISTS orders(order_id INTEGER PRIMARY KEY,
                                  at TEXT DEFAULT (datetime('now','localtime')));
CREATE TABLE IF NOT EXISTS order_items(order_id INTEGER REFERENCES orders(order_id),
                                       item TEXT, price INTEGER);""")

def submit_order(basket):
    if not basket:
        return basket, pd.DataFrame(columns=["品項","價格"]), "⚠️ 籃子是空的"
    with ocon:                                            # 一張單＋N 列明細：同生共死
        oid = ocon.execute("INSERT INTO orders DEFAULT VALUES").lastrowid
        ocon.executemany("INSERT INTO order_items VALUES (?,?,?)",
                         [(oid, item, price) for item, price in basket])
    return [], pd.DataFrame(columns=["品項","價格"]), f"✅ 訂單 #{oid} 成立"

# Blocks 裡加：gr.Button("送出訂單", variant="primary").click(submit_order, [basket], [basket, tbl, subtotal])
```
「State 暫存 → 按下送出 → 交易落庫」正是點餐、購票、下單類系統的共同骨架（lastrowid 串起單頭與明細）。
</details>

In [ ]:
# 練習 E 工作區
# TODO




In [ ]:
# 招式三 .select：點 Dataframe 的一列 → 知道選到誰 → 刪除（編輯同理：把值填回表單）
def member_table():
    return pd.read_sql_query("SELECT member_id, name, dept, balance FROM member ORDER BY member_id", gcon)

def delete_member(member_id):
    with gcon:
        n = gcon.execute("DELETE FROM member WHERE member_id = ?", (int(member_id),)).rowcount
    return ("🗑 已刪除" if n else "⚠️ 沒有這個編號"), member_table()

def on_select(evt: gr.SelectData, cur_table):          # 型別註記 gr.SelectData → Gradio 自動塞事件資訊
    row = cur_table.iloc[evt.index[0]]                 # evt.index = [列號, 欄號]
    return int(row["member_id"]), f"選到 #{int(row['member_id'])} {row['name']}"

with gr.Blocks() as manage_ui:
    gr.Markdown("### 點一列選取，再按刪除")
    tbl = gr.Dataframe(value=member_table(), interactive=False)
    selected = gr.Number(value=0, visible=False)        # 藏起來的「目前選中」暫存
    msg2 = gr.Textbox(interactive=False, label="狀態")
    tbl.select(on_select, [tbl], [selected, msg2])
    gr.Button("刪除選中", variant="stop").click(delete_member, [selected], [msg2, tbl])

n0 = len(member_table()); m, _ = delete_member(999999)  # 底層函數先測
assert "沒有" in m and len(member_table()) == n0
print("✅ 刪除邏輯測試通過（UI 的 .select 只是把「哪一列」交給後端）")

In [ ]:
# 錯誤回饋的三個層次：訊息欄（低調）→ gr.Warning（提醒橫幅）→ gr.Error（紅色擋下）
def topup_check(amt):
    if amt is None or amt <= 0:
        raise gr.Error("金額必須是正數")        # 紅色錯誤框，事件中止（別讓 traceback 噴給使用者）
    if amt > 10000:
        gr.Warning("金額異常地大，確定嗎？")    # 黃色提醒，事件照常執行
    return f"✅ 收到儲值 {int(amt)} 元"

with gr.Blocks() as feedback_demo:
    a = gr.Number(label="儲值金額", value=100)
    o = gr.Textbox(interactive=False, label="結果")
    gr.Button("儲值").click(topup_check, a, o)

print("三層回饋：日常結果用訊息欄；可疑但放行用 Warning；必須擋下用 gr.Error。")
print("後端純函數仍以「回傳訊息字串」為主——gr.Error 只包在 UI 層函數，底層才好測試。")

In [ ]:
# 報表分頁的關鍵元件：gr.Plot（把 matplotlib 圖嵌進介面）——專題「≥1 張圖表」就靠它
def balance_chart():
    df = list_members()
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(df["姓名"], df["餘額"])
    ax.set_ylabel("餘額"); ax.set_title("會員餘額一覽")      # 中文標籤 OK：字型 bootstrap 在 1.6 跑過了
    plt.tight_layout()
    return fig                       # 直接回傳 matplotlib figure，gr.Plot 會顯示它

fig = balance_chart(); print("圖產生成功：", type(fig).__name__)
plt.close(fig)

with gr.Blocks() as report_ui:
    gr.Markdown("## 統計報表")
    plot = gr.Plot(label="會員餘額")
    gr.Button("重新整理").click(balance_chart, outputs=plot)
print("報表分頁建好（Plot 元件）")

In [ ]:
# 報表匯出：查詢結果存成 CSV 檔案，介面上給一顆下載鈕（老師、業主最常要的功能）
def export_members_csv():
    path = "members_export.csv"
    member_table().to_csv(path, index=False)
    return path                                   # 回傳「檔案路徑」→ gr.File / DownloadButton 變成可下載

p = export_members_csv()
import os
print("匯出測試：", p, os.path.getsize(p), "bytes ✅")

with gr.Blocks() as export_ui:
    gr.Markdown("### 報表匯出")
    f = gr.File(label="下載檔案")
    gr.Button("匯出會員 CSV").click(export_members_csv, outputs=f)
print("→ pattern：後端函數回傳檔案路徑即可。Excel 派的業主給 CSV 最保險（Excel 直接開）。")

In [ ]:
# 順手一招：gr.Markdown 也能當「輸出」——資料庫撈公告，動態組 markdown（首頁公告欄 pattern）
def news_md():
    rows = gcon.execute("SELECT body, at FROM announcement ORDER BY id DESC").fetchall()
    return "### 📢 最新公告\n" + "\n".join(f"- {r['body']}（{r['at'][:16]}）" for r in rows)

print(news_md())
with gr.Blocks() as news_ui:
    news_box = gr.Markdown(news_md())
    gr.Button("重新整理").click(news_md, outputs=news_box)
print("→ 不是所有輸出都是表格：公告、操作說明、統計摘要一句話，用 Markdown 元件最順眼。")

## 2.7 分頁：`gr.Tab`——專題要求的「≥3 分頁」骨架

```python
with gr.Blocks(title="我的系統") as app:
    with gr.Tab("日常操作"):   ...    # 報名 / 點餐 / 借書 ...
    with gr.Tab("後台管理"):   ...    # 新增 / 修改 / 刪除
    with gr.Tab("統計報表"):   ...    # 查詢 + gr.Dataframe + gr.Plot
app.launch()
```

**三分頁的分工哲學**：「日常操作」給一般使用者（流程化、防呆）、「後台管理」給管理員（CRUD 全開）、
「統計報表」給決策者（唯讀、圖表）。同一顆資料庫，三種視角——這個劃分本身就是設計。

下一格把今天的零件組成一個**完整應用骨架**——這就是你專題 notebook 最後那顆 `app.launch()` 的樣板。

In [ ]:
# 完整應用骨架：三分頁（查詢 / 後台 / 報表）——直接當你專題的起手式
with gr.Blocks(title="會員系統 Demo") as app:
    gr.Markdown("# 會員系統（U05 完整骨架示範）")
    with gr.Tab("🔍 查詢"):
        kw  = gr.Textbox(label="姓名關鍵字")
        bal = gr.Slider(0, 100, value=0, step=10, label="最低餘額")
        out = gr.Dataframe()
        kw.change(search_members, [kw, bal], out); bal.change(search_members, [kw, bal], out)
    with gr.Tab("⚙️ 後台"):
        n = gr.Textbox(label="姓名"); d = gr.Dropdown(["統計","資訊","數學","企管"], label="系所")
        b = gr.Number(label="餘額", value=0); m = gr.Textbox(label="訊息", interactive=False)
        t = gr.Dataframe(value=list_members())
        gr.Button("新增", variant="primary").click(add_member, [n, d, b], [m, t])
    with gr.Tab("📊 報表"):
        p = gr.Plot(value=balance_chart())
        gr.Button("重新整理").click(balance_chart, outputs=p)

print("✅ 完整三分頁 app 組裝完成；下一格 launch。")

In [ ]:
# [SKIP-TEST] 啟動三分頁骨架
app.launch(height=600)   # 報告時：app.launch(share=True)

### 關於 Colab 內嵌與 `share=True`

- `launch()` 在 Colab 會把介面**內嵌**在輸出格；同一時間跑多個 app 沒問題（各佔一格）。
- `share=True` 生出一個 `https://xxx.gradio.live` 公開網址（72 小時有效）：**報告日讓全班手機操作就靠它**（問卷類題目現場蒐集回覆的 demo 亮點）。
- Runtime 一斷，網址就死——所以 demo 前重跑全部、備援截圖（共同要求第 10 條）。
- 改了函數要**重跑該格**（重新 launch）介面才會用到新版。

### Gradio 翻車 FAQ（課堂實作卡住先看這）

| 症狀 | 原因與解法 |
|---|---|
| `SQLite objects created in a thread...` | 連線忘了 `check_same_thread=False`（2.4 紀律 1） |
| 按了按鈕沒反應／用到舊版函數 | 改了函數沒**重跑定義格＋重新 launch** |
| Dataframe 顯示數字變 `1.0` | pandas 把整數欄升成 float（有 NaN 時）——顯示前 `astype` 或 `fillna` |
| 下拉選單是舊清單 | 忘了把「重撈選單」放進 outputs（2.6 招式一） |
| `share=True` 連結打不開 | 72 小時過期或 runtime 斷了——重跑 launch 拿新網址 |
| 介面出現但操作報錯 | 看 Colab 輸出格的 traceback——UI 層 `raise gr.Error`，底層回傳訊息字串 |

## 2.8 第二個完整範例：福利社儲值卡小店 🏪

把今天所有零件組成一個「**共同要求縮小版**」（非指派題目，放心參考）：
4 張表（card／product／purchase／purchase_item）、交易保護的購買（扣庫存＋扣款＋寫單，同生共死）、
三分頁、報表圖。**看懂這個，你的專題就是它的放大版。**

In [ ]:
# 福利社（1/3）：schema 與種子資料——注意 purchase_item 存「成交當下的價」（U04 教的歷史快照）
sh = sqlite3.connect("shop.db", check_same_thread=False)
sh.row_factory = sqlite3.Row
sh.execute("PRAGMA foreign_keys = ON")
sh.executescript("""
DROP TABLE IF EXISTS purchase_item; DROP TABLE IF EXISTS purchase;
DROP TABLE IF EXISTS product; DROP TABLE IF EXISTS card;
CREATE TABLE card(
  card_id INTEGER PRIMARY KEY,
  owner   TEXT NOT NULL,
  balance INTEGER NOT NULL DEFAULT 0 CHECK (balance >= 0));
CREATE TABLE product(
  pid   INTEGER PRIMARY KEY,
  pname TEXT NOT NULL UNIQUE,
  price INTEGER NOT NULL CHECK (price > 0),
  stock INTEGER NOT NULL DEFAULT 0 CHECK (stock >= 0));
CREATE TABLE purchase(
  purchase_id INTEGER PRIMARY KEY,
  card_id INTEGER NOT NULL REFERENCES card(card_id),
  at TEXT NOT NULL DEFAULT (datetime('now','localtime')));
CREATE TABLE purchase_item(
  purchase_id INTEGER NOT NULL REFERENCES purchase(purchase_id) ON DELETE CASCADE,
  pid INTEGER NOT NULL REFERENCES product(pid),
  qty INTEGER NOT NULL CHECK (qty > 0),
  unit_price INTEGER NOT NULL,                  -- 成交當下的價格（歷史證據，不是重複）
  PRIMARY KEY (purchase_id, pid));              -- 弱實體：複合主鍵＋CASCADE
""")
sh.executemany("INSERT INTO card(owner, balance) VALUES (?,?)",
               [("林佳蓉", 500), ("陳威廷", 120), ("吳孟軒", 60)])
sh.executemany("INSERT INTO product(pname, price, stock) VALUES (?,?,?)",
               [("咖啡", 45, 10), ("餅乾", 25, 8), ("泡麵", 38, 5), ("能量飲", 55, 2)])
sh.commit()
print("shop.db 就緒 ✅（4 表；兩條 CHECK 分別守住餘額與庫存不為負）")

In [ ]:
# 福利社（2/3）：核心業務函數「購買」——扣庫存＋扣款＋寫單，一個交易同生共死
def buy(card_id, pname, qty):
    try:
        with sh:
            p = sh.execute("SELECT pid, price, stock FROM product WHERE pname = ?", (pname,)).fetchone()
            if p is None:
                raise ValueError("沒有這個商品")
            total = p["price"] * qty
            n_upd = sh.execute("""UPDATE product SET stock = stock - ?
                                   WHERE pid = ? AND stock >= ?""",       # 條件式 UPDATE（U06 細講它防競態）
                               (qty, p["pid"], qty)).rowcount
            if n_upd == 0:
                raise ValueError(f"庫存不足（只剩 {p['stock']}）")
            sh.execute("UPDATE card SET balance = balance - ? WHERE card_id = ?", (total, card_id))
            cur2 = sh.execute("INSERT INTO purchase(card_id) VALUES (?)", (card_id,))
            sh.execute("INSERT INTO purchase_item VALUES (?,?,?,?)",
                       (cur2.lastrowid, p["pid"], qty, p["price"]))
        return f"✅ 購買成功：{pname} × {qty} ＝ {total} 元"
    except ValueError as e:
        return f"❌ {e}（整筆取消，什麼都沒動）"
    except sqlite3.IntegrityError:
        return "❌ 餘額不足（整筆取消：庫存也原封退回）"

print(buy(1, "咖啡", 2))          # 成功
print(buy(3, "能量飲", 5))        # 庫存只有 2 → 擋
print(buy(3, "咖啡", 2))          # 60 元的卡買 90 元 → balance CHECK 擋，連剛扣的庫存都回滾

bal_map = {r["owner"]: r["balance"] for r in sh.execute("SELECT owner, balance FROM card")}
stock = sh.execute("SELECT stock FROM product WHERE pname = '咖啡'").fetchone()[0]
print("餘額：", bal_map, "・咖啡庫存：", stock)
assert bal_map["吳孟軒"] == 60 and stock == 8      # 兩次失敗都「整筆」沒發生
print("✅ 交易保護測試通過——三件事同生共死，失敗就像沒發生過")

In [ ]:
# 福利社（3/3）：報表函數 ＋ 三分頁組裝
def card_list():
    return pd.read_sql_query("SELECT card_id, owner, balance FROM card ORDER BY card_id", sh)

def product_list():
    return pd.read_sql_query("SELECT pid, pname, price, stock FROM product ORDER BY pid", sh)

def bestseller_chart():
    df = pd.read_sql_query("""SELECT p.pname, SUM(i.qty) AS n
                              FROM purchase_item i JOIN product p ON i.pid = p.pid
                              GROUP BY p.pid ORDER BY n DESC""", sh)
    fig, ax = plt.subplots(figsize=(6, 2.8))
    ax.bar(df.pname, df.n); ax.set_title("熱銷排行"); ax.set_ylabel("銷售數量")
    plt.tight_layout(); return fig

def buy_and_refresh(card_id, pname, qty):
    return buy(int(card_id), pname, int(qty)), card_list(), product_list()

def topup(card_id, amt):
    if not amt or amt <= 0:
        return "⚠️ 金額要是正數", card_list()
    with sh:
        sh.execute("UPDATE card SET balance = balance + ? WHERE card_id = ?", (int(amt), int(card_id)))
    return f"✅ 已儲值 {int(amt)} 元", card_list()

fig = bestseller_chart(); plt.close(fig)          # 報表函數先測
assert topup(2, 100)[0].startswith("✅") and topup(2, -5)[0].startswith("⚠️")

with gr.Blocks(title="福利社儲值卡") as shop_ui:
    gr.Markdown("# 🏪 福利社儲值卡小店（U05 完整範例）")
    with gr.Tab("🧋 購買"):
        dd_card = gr.Dropdown(choices=[(f"{r['owner']}（#{r['card_id']}）", r["card_id"])
                                       for r in sh.execute("SELECT card_id, owner FROM card")],
                              value=1, label="卡片")
        dd_prod = gr.Dropdown(choices=[r["pname"] for r in sh.execute("SELECT pname FROM product")],
                              value="咖啡", label="商品")
        qty_sl = gr.Slider(1, 5, value=1, step=1, label="數量")
        msg3 = gr.Textbox(interactive=False, label="結果")
        tbl_cards = gr.Dataframe(value=card_list(), label="卡片餘額")
        tbl_prods = gr.Dataframe(value=product_list(), label="商品庫存")
        gr.Button("購買", variant="primary").click(buy_and_refresh, [dd_card, dd_prod, qty_sl],
                                                   [msg3, tbl_cards, tbl_prods])
    with gr.Tab("⚙️ 後台"):
        card_in = gr.Number(label="card_id", value=1); amt_in = gr.Number(label="儲值金額", value=100)
        msg4 = gr.Textbox(interactive=False, label="訊息"); tbl_cards2 = gr.Dataframe(value=card_list())
        gr.Button("儲值").click(topup, [card_in, amt_in], [msg4, tbl_cards2])
    with gr.Tab("📊 報表"):
        gr.Plot(value=bestseller_chart())

print("✅ 福利社三分頁組裝完成——「共同要求」的縮小版都在這；下一格 launch")

In [ ]:
# [SKIP-TEST] 福利社開店！
shop_ui.launch(height=680)

### 隨堂練習 F（3 分鐘，口頭＋筆記）：福利社對照共同要求

翻開 `projects.md` §3 的 10 條，逐條問「福利社做到了嗎？」——找出**沒做到的三條**。

<details><summary>答案</summary>
沒做到：**2**（只有幾筆種子資料，沒有萬列合成）、**5**（報表只有 1 張，沒 window）、**6**（沒有索引效能對照）——
外加 4 的「併發競態示範」只有防護沒有重現（U06 補）。所以福利社＝縮小版；把這四塊補滿、換成你的題目，就是完整專題。
</details>

### 隨堂練習 G（3 分鐘，紙上）：幫福利社設計「退貨」

規格思考題：`refund(purchase_id)` 要在**一個交易**裡做哪幾件事？哪些情況要擋？

<details><summary>參考答案</summary>
三件事同生共死：① 庫存加回（每個 purchase_item 的 qty）② 卡片餘額加回（Σ qty×unit_price——用**成交價**不是現價！U04 歷史快照的用途在這現形）③ 標記或刪除購買紀錄（留審計軌跡的話用 status='退貨' 而不是 DELETE）。
要擋：查無此單、已退過的單（狀態機！）。——這種「反向操作」是口試常客：它逼你把交易、歷史快照、狀態機三招一起用。
</details>

## 課堂實作：把你的資料層接上第一個 Gradio 分頁

帶著 U04 工作坊的 DDL，照三步走（卡住先翻福利社範例，再問 AI——貼 schema＋函數簽名）：

1. **（10 分）建庫**：把你的 `CREATE TABLE` 貼進下一格跑起來，灌 3–5 筆示範資料。
2. **（10 分）資料層**：寫兩個函數——`add_x(...)`（`with con:`＋友善錯誤）與 `search_x(...)`（回傳 DataFrame）；**各配一個 assert**。
3. **（15 分）接 UI**：模仿 2.5 的骨架包成一個分頁，`launch()` 起來按按看。

做到這裡，你的專題已經「會動」了——下個單元把它長成完整 CRUD＋擋住併發競態。

In [ ]:
# 課堂實作工作區
mycon = sqlite3.connect("myapp.db", check_same_thread=False)
mycon.row_factory = sqlite3.Row
mycon.execute("PRAGMA foreign_keys = ON")

# TODO 1：你的 DDL（mycon.executescript(...)）＋示範資料
# TODO 2：def add_x(...): ...　def search_x(...): ...　＋ assert
# TODO 3：with gr.Blocks() as myapp: ...  然後 myapp.launch()

print("工作區就緒——貼上你的 schema 開工")

# ★ 專題說明會（35 分鐘）

## 共同要求 10 條（`projects.md` §3，這是評分主體，人人一樣）

| # | 要求 | 這單元學了哪部分 |
|---|---|---|
| 1 | Schema：≥4 表、3NF、PK/FK＋≥3 種約束、附圖 | U04 已做（工作坊） |
| 2 | 合成 ≥10,000 列、固定 seed、分佈擬真 | ✅ 今天 1.6 七講究 |
| 3 | CRUD 全套、查詢用 `?` 傳值 | ✅ 今天 1.3、2.5–2.6 |
| 4 | 交易保護 ＋ 一個併發競態示範 | 交易 ✅ 今天 1.4；競態 U06 |
| 5 | 報表 ≥5（≥1 window、≥2 join、≥1 圖） | U03 查詢 ＋ 今天 gr.Plot |
| 6 | 最慢查詢的索引前後效能對照 | U07 |
| 7 | Gradio ≥3 分頁、結果用 gr.Dataframe | ✅ 今天 2.7–2.8 |
| 8 | ≥8 個 assert（含 2 個「應該失敗」） | ✅ 今天整節都在示範 |
| 9 | AI 使用說明 | 每單元【AI 協作】都在練 |
| 10 | 15 分鐘簡報＋live demo＋備援 | U09 報告規範 |

**你已經有能力完成 1–5、7–9 的大半了。** 剩下的 U06（競態）、U07（索引）補齊，就是完整專題。

### 現場自評（3 分鐘，拿出你的專題現況對照上表）

每條標 ✅（已完成）／🔜（知道怎麼做）／❌（還不會）——然後把三個最要緊的 ❌ 寫成「下一步」：

- ❌ 最常見的三個：**競態**（U06 下次就教）、**索引對照**（U07）、**備援**（報告週前做）。
- 🔜 累積很快的兩個：**assert**（每寫一個函數配一個）、**報表**（U03 的 lab 換成你的資料再寫一次）。
- 今天課後最划算的一步：把福利社範例改名成你的題目，表換成你的表——骨架直接活。

## 兩支示範影片（教師專題，非指派題目——當完成度標竿，不是抄襲對象）

1. **圖書館借閱管理系統**：book/member/loan/reservation 四表；借出（庫存交易）、還書、續借、逾期查詢、預約候補、熱門書排行報表。→ 對應「借還／流轉」類題型。
2. **個人記帳分析 MoneyBook**：account/category/transaction/budget；記帳 CRUD、預算超支警示、月報表與分類佔比圖、window 算收支趨勢。→ 對應「流水帳＋分析」類題型。

**看影片時請注意**（這些就是評分官在看的）：
- schema 怎麼對應情境；哪些規則用約束、哪些用交易、哪些用程式邏輯
- demo 的「腳本化路徑」怎麼走得順（不是即興亂點）
- 報表怎麼從資料裡「講出一個故事」
- 完成度的手感：一個「做完了」的專題長什麼樣

程式完整公開在 repo `demo/library/` 與 `demo/moneybook/`（含逐格註解），回家可整本重跑、逐段拆解。

## AI 協作工作流建議（把整學期學的串起來）

```
1. 設計   ── 貼題目情境 → AI 給 ER/DDL → 你跑 U04 的驗證流程（closure、異常踩點、自檢器）
2. 資料   ── 描述使用者行為 → AI 給合成腳本 → 你檢查分佈是否擬真（長尾？節律？相關？）
3. 後端   ── 一次要一個函數（「寫一個借書函數，用交易，庫存不足要擋」）→ 你寫 assert 驗收
4. 前端   ── 「用 Gradio Blocks 把這三個函數做成分頁介面」→ 你調版面
5. 除錯   ── 貼完整錯誤訊息 → AI 解釋 → 你理解後才套用（別盲貼）
6. 反質詢 ── 「這個 schema 撐得住『兩人同時報名』嗎？」（U06 見真章）
             「這句查詢十萬列時會不會慢？」（U07 用 EXPLAIN 對質）
             「同一個數字用 pandas 再算一次對照」（U03 的雙引擎交叉驗證）
```

**紅線**：口試任指一段你要能解釋；繳交附 AI 使用說明。**AI 寫得快，你負責寫得對。**

## 專題進度建議（非繳交）

對照 [syllabus](https://github.com/chang-ye-tu/db/blob/master/syllabus.md) 進度表：**到 U05，建議完成「schema 定稿、建表、合成萬列資料、資料層函數」**：

1. U04 的 DDL 定稿（自檢器與踩點測試都綠）；
2. 用今天 1.6 的手法**合成 ≥10,000 列**（固定 seed；長尾／節律／右偏／相關至少各用一招），markdown 記下生成假設（使用者是誰、行為規律是什麼），並用練習 B 的兩張圖自我驗收；
3. **資料層函數 ≥6 個**（純函數、`?` 傳值、業務操作包 `with con:`）；
4. **測試開始累積**：目標 ≥8 個 assert、含 ≥2 個「應該失敗」（約束違反、餘額／庫存不足）；
5. 順手把「課堂實作」那個分頁完成——你的 app 已經會動了。

**不用繳交**——但示範影片裡的完成度，就是這樣一步一塊疊出來的。下個單元課堂實作：完整 CRUD 分頁＋擋住併發競態。

# 本單元你應該帶走

1. sqlite3 全套：`Row` 用欄名、fetch 家族、`lastrowid`／`rowcount`、`executescript` 放 setup、date adapter/converter 自己註冊（3.12 起的正規做法）。
2. **`?` 佔位符是把值交給 SQL 的正常寫法**：引號、日期、None 都不用煩惱，`executemany` 也靠它；動態 IN 產生剛好數量的 `?`；欄名／表名用白名單挑。
3. 交易：`with con:` 讓多步操作同生共死；**savepoint** 讓批次匯入「壞一包退一包」；UPSERT × executemany 重匯不炸。
4. 匯入用 executemany＋單一交易（差幾十倍）；合成資料是統計建模——長尾、節律、右偏、**相關欄位、小時節律、假名單、髒資料注入**，seed 與假設寫進報告、畫圖自我驗收。
5. Gradio：`Interface` 快、`Blocks` 真；事件三兄弟；**`gr.update` 刷新下拉、`gr.State` 暫存、`.select` 列選取**；三層錯誤回饋＋CSV 匯出；**UI 薄、邏輯厚**——後端純函數先 assert 再接 UI。
6. 福利社小店＝共同要求縮小版（練習 F 找到了它缺的三條）：**回家把它改成你的題目**，就是專題進度。

**下個單元**：完整應用模式（多分頁 CRUD ＋ 併發競態防護）＋ 資料庫內幕第一站：儲存引擎（親眼看資料怎麼落地）。讀物：sqlite3 官方文件、Gradio Quickstart；Silberschatz ch12–13 預習。

---
## 附錄 A：Gradio ＋ sqlite3 專題 cheatsheet

```python
# 連線（放 notebook 開頭）
import sqlite3, pandas as pd, gradio as gr
con = sqlite3.connect("project.db", check_same_thread=False)
con.row_factory = sqlite3.Row
con.execute("PRAGMA foreign_keys = ON")

# 後端函數範本（純函數、? 傳值、可測試）
def do_something(a, b):
    a = (a or "").strip()
    if not a: return "⚠️ 必填", load_table()
    try:
        with con:                              # 交易：同生共死
            con.execute("INSERT INTO t(x, y) VALUES (?,?)", (a, b))
        return "✅ 成功", load_table()
    except sqlite3.IntegrityError as e:
        return f"❌ {e}", load_table()

def load_table():
    return pd.read_sql_query("SELECT ... FROM t ORDER BY id", con)

# 佔位符
con.execute("... WHERE x = ? AND y = ?", (a, b))          # 位置版
con.execute("... VALUES (:x, :y)", {"x": a, "y": b})      # 具名版（表單 dict 直送）
sql = f"... IN ({','.join('?' * len(ids))})"              # 動態 IN
# 欄名要動態 → 白名單挑好再組；SAVEPOINT sp1 / ROLLBACK TO sp1 / RELEASE sp1

# 前端骨架
with gr.Blocks(title="系統名") as app:
    with gr.Tab("操作"):
        x = gr.Textbox(label="..."); y = gr.Number(label="...")
        msg = gr.Textbox(interactive=False); tbl = gr.Dataframe(value=load_table())
        gr.Button("送出", variant="primary").click(do_something, [x, y], [msg, tbl])
    with gr.Tab("報表"):
        gr.Plot(value=make_chart())
app.launch()          # 報告：app.launch(share=True)

# 常用元件與事件
gr.Textbox / Number / Slider / Dropdown / Radio / Checkbox / Button / Dataframe / Plot / State / File
.click() .change() .submit() .select(fn(evt: gr.SelectData, ...))
gr.update(choices=..., value=...)       # 更新元件屬性（下拉即時刷新）
raise gr.Error("訊息")；gr.Warning("訊息")   # UI 層回饋
demo.load(fn, outputs=...)              # 頁面載入時先跑一次
```

## 附錄 B：讀物地圖（本單元）

| 主題 | 去哪讀 |
|---|---|
| sqlite3 API（Row／交易／adapter） | Python 官方文件 `sqlite3` 模組（how-to 段有 placeholder 與 adapter 範例） |
| SQLite 交易與 savepoint | https://sqlite.org/lang_transaction.html ・ https://sqlite.org/lang_savepoint.html |
| Gradio 入門 | Quickstart https://www.gradio.app/guides/quickstart |
| Gradio Blocks 與事件 | https://www.gradio.app/guides/blocks-and-event-listeners |
| Dataframe 列選取／State | Gradio Docs：`gr.SelectData`、`gr.State` |

下個單元預習（內幕第一站）：Silberschatz ch12–13（storage）；https://sqlite.org/fileformat2.html 掃一眼就好。